# Groceries - Data Preparation and Transformation with Restrictions

This file is used to prepare and transform the groceries data. The following restrictions apply to this file:

## Restrictions:

1. Baskets with less than 30 items
2. Customers with less than 5 baskets

## Main Process and Steps:

### 1. Parameter Setup and Data Overview:
Set up the packages, path, and dataset name. It also includes an overview of the data, such as the date period of the data.

### 2. Indexing of Items:
Create indices to represent the items and a mapping table for reference.

### 3. Indexing of Customers:
Create indices to represent the customer numbers and a mapping table for reference.

### 4. Baskets for Each Customer and Items in Each Basket:
By utilizing timestamps, it is possible to determine which items were purchased together, group them into baskets, and identify the number of baskets each customer has.

### 5. Apply Restrictions:
Apply the specified restrictions to the data.

### 6. Separate Train and Test Datasets:
Split the dataset so that the number of baskets in the training set is equal to the number of baskets in the testing set. If a customer has an odd number of baskets, delete the latest date (last row).

### 7. Generate 3 Files as Input for the Model:
Generate the following files: `train_u2b.txt`, `train_b2i.txt`, `test_b2i.txt`and `test_u2b.txt`.

## Coding

### 1. Parameter Setup and Data Overview:

In [1]:
# Import packages
import pandas as pd
import numpy as np

import random 

In [2]:
# Set and read dataset
path_name = '/Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres'

df = pd.read_csv(path_name + '/Groceries_dataset.csv')
df.head(5)

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [3]:
# The date period of the data
# Convert 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Find first and last dates of data
first_date = df['Date'].min()
last_date = df['Date'].max()

print(f"First date in the dataset: {first_date}")
print(f"Last date in the dataset: {last_date}")

First date in the dataset: 2014-01-01 00:00:00
Last date in the dataset: 2015-12-30 00:00:00


### 2. Indexing of Items:

In [4]:
# Factorize the itemDescription column
df['item_no'], item_labels = pd.factorize(df['itemDescription'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'item_no': range(len(item_labels)),
    'itemDescription': item_labels
})

item_mapping.to_csv(path_name+r'\grocerieswithresname\item_index_mapping.txt', sep='\t', index=False)
# Drop the original itemDescription column
df = df.drop(columns=['itemDescription'])
df

,Member_number,Date,item_no
0,1808,2015-07-21,0
1,2552,2015-01-05,1
2,2300,2015-09-19,2
3,1187,2015-12-12,3
4,3037,2015-02-01,1
...,...,...,...
38760,4471,2014-10-08,75
38761,2022,2014-02-23,64
38762,1097,2014-04-16,153
38763,1510,2014-12-03,11


### 3. Indexing of Customers:

In [5]:
# Factorize the Member_number column
df['uid'], item_labels = pd.factorize(df['Member_number'])

# Create a mapping table
item_mapping = pd.DataFrame({
    'Member_number': item_labels,
    'uid': range(len(item_labels))
})

# Drop the original Member_number column
df = df.drop(columns=['Member_number'])
df

,Date,item_no,uid
0,2015-07-21,0,0
1,2015-01-05,1,1
2,2015-09-19,2,2
3,2015-12-12,3,3
4,2015-02-01,1,4
...,...,...,...
38760,2014-10-08,75,1154
38761,2014-02-23,64,81
38762,2014-04-16,153,2768
38763,2014-12-03,11,301


### 4. Baskets for Each Customer and Items in Each Basket:

In [42]:
# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Group data by 'uid' and 'Date' to create baskets for each customer and items in each basket
grouped_df = df.groupby(['uid', 'Date'])['item_no'].apply(list).reset_index().sort_values(by=['uid', 'Date'])

expanded_df = grouped_df['item_no'].apply(pd.Series).rename(columns=lambda x: str(x+1))
expanded_df = expanded_df.fillna('')

for col in expanded_df.columns[:]:
    expanded_df[col] = expanded_df[col].apply(lambda x: str(int(x)) if x != '' else x)

final_df = pd.concat([grouped_df[['uid', 'Date']], expanded_df], axis=1) 

print("Columns in final_df:", final_df.columns)
print(final_df.head())

Empty DataFrame
Columns: [uid, Date, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, basket_number]
Index: []
Basket 13 count: 0
Columns in final_df: Index(['uid', 'Date', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11',
       'basket_number'],
      dtype='object')
   uid       Date    1    2    3 4 5 6 7 8 9 10 11  basket_number
0    0 2014-11-29   56    1                                     0
1    0 2014-12-15   34    6  113                                1
2    0 2015-02-04  118  102                                     2
3    0 2015-07-21    0    4   64                                3
4    1 2014-02-19    3   13                                     0


### 5. Apply Restrictions:

In [8]:
# Determine the item columns, excluding 'uid' and 'Date'
item_columns = [col for col in final_df.columns if col not in ['uid', 'Date']]
print("Item columns:", item_columns)

# Calculate the count of valid items for each row
final_df['valid_count'] = final_df[item_columns].apply(
    lambda row: row.notna().sum(), axis=1
)

# Filter baskets where the number of valid items is <= 31
final_df = final_df.query('valid_count <= 31')
print("Columns in final_df after filtering:", final_df.columns)

# Function to trim item columns to a specified maximum count
def trim_columns(row, max_count):
    """
    Trim a row to include only valid non-empty values up to max_count.
    Fill the remaining columns with empty strings ('') if needed.
    """
    # Filter out empty or NaN values
    valid_values = [x for x in row if pd.notna(x) and x != '']
    # Trim to max_count and pad with empty strings
    trimmed = valid_values[:max_count]
    return trimmed + [''] * (max_count - len(trimmed))

# Determine the maximum count of valid items across all baskets
max_valid_count = final_df['valid_count'].max()
print("Max valid count:", max_valid_count)

# Apply trimming to item columns
trimmed_data = final_df[item_columns].apply(
    lambda row: trim_columns(row, max_valid_count), axis=1
)

# Create a new DataFrame for the trimmed item columns
df_trimmed = pd.DataFrame(
    trimmed_data.tolist(),
    index=final_df.index,
    columns=[f'item_{i+1}' for i in range(max_valid_count)]
)
print("Columns in df_trimmed:", df_trimmed.columns)

# Concatenate the key columns ('uid' and 'Date') with the trimmed item columns
final_df = pd.concat([final_df[['uid', 'Date']].reset_index(drop=True), df_trimmed], axis=1)
print("Columns in final_df after concat:", final_df.columns)

# Drop the auxiliary 'valid_count' column
final_df = final_df.drop(columns=['valid_count'], errors='ignore')

# Verify the resulting DataFrame
print(final_df.head())

Item columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', 'basket_number']
Columns in final_df after filtering: Index(['uid', 'Date', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11',
       'basket_number', 'valid_count'],
      dtype='object')
Max valid count: 12
Columns in df_trimmed: Index(['item_1', 'item_2', 'item_3', 'item_4', 'item_5', 'item_6', 'item_7',
       'item_8', 'item_9', 'item_10', 'item_11', 'item_12'],
      dtype='object')
Columns in final_df after concat: Index(['uid', 'Date', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5',
       'item_6', 'item_7', 'item_8', 'item_9', 'item_10', 'item_11',
       'item_12'],
      dtype='object')
   uid       Date item_1 item_2 item_3 item_4 item_5 item_6 item_7 item_8  \
0    0 2014-11-29     56      1      0                                      
1    0 2014-12-15     34      6    113      1                               
2    0 2015-02-04    118    102      2                                      
3    0

In [9]:
# Customers with less than 5 baskets
# Keep the first 10 rows
final_df = final_df.groupby('uid', group_keys=False).apply(
    lambda x: x.sort_values(by='Date').head(10)
).reset_index(drop=True)
final_df

/var/folders/7m/7wnk_cjx3t33tp2_97x56zj40000gn/T/ipykernel_11564/1008573384.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = final_df.groupby('uid', group_keys=False).apply(


,uid,Date,item_1,item_2,item_3,item_4,item_5,item_6,item_7,item_8,item_9,item_10,item_11,item_12
0,0,2014-11-29,56,1,0,,,,,,,,,
1,0,2014-12-15,34,6,113,1,,,,,,,,
2,0,2015-02-04,118,102,2,,,,,,,,,
3,0,2015-07-21,0,4,64,3,,,,,,,,
4,1,2014-02-19,3,13,0,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14953,3893,2014-08-19,64,37,0,,,,,,,,,
14954,3894,2014-10-20,37,17,0,,,,,,,,,
14955,3895,2014-02-22,75,69,0,,,,,,,,,
14956,3896,2014-05-23,71,141,0,,,,,,,,,


### 6. Separate Train and Test Datasets:

In [10]:
# Split function that skips uid and Date but keeps them in output
def split_basket_items(row, train_ratio=0.8):
    uid = row['uid']
    date = row['Date']
    
    # Only keep item columns (skip uid and Date)
    items = row[2:].tolist()
    items = [int(item) for item in items if item != '']  # Remove blanks and convert to int

    if len(items) == 0:
        return [], []

    if len(items) == 1:
        return items, []

    try:
        test_size = max(1, int(len(items) * (1 - train_ratio)))
        test_items = random.sample(items, test_size)
        train_items = [item for item in items if item not in test_items]
        return train_items, test_items
    except Exception as e:
        print(f" Error in basket (uid: {uid}): {e}")
        return [], []



In [11]:
# Apply the function row by row
train_rows = []
test_rows = []
basket_number = 0

for idx, row in final_df.iterrows():
    train_items, test_items = split_basket_items(row)
    uid = row['uid']
    date = row['Date']

    if train_items:
        train_rows.append(
            [uid, date] + train_items +
            [''] * (final_df.shape[1] - 2 - len(train_items)) +
            [basket_number]
        )

    if test_items:
        test_rows.append(
            [uid, date] + test_items +
            [''] * (final_df.shape[1] - 2 - len(test_items)) +
            [basket_number]
        )

    basket_number += 1

# Build final train/test DataFrames with proper column names
train_df = pd.DataFrame(train_rows, columns=list(final_df.columns) + ['basket_number'])
test_df = pd.DataFrame(test_rows, columns=list(final_df.columns) + ['basket_number'])

In [12]:
print("Train sample:")
print(train_df.head())

Train sample:
   uid       Date  item_1 item_2 item_3 item_4 item_5 item_6 item_7 item_8  \
0    0 2014-11-29      56      1                                             
1    0 2014-12-15      34      6      1                                      
2    0 2015-02-04     118      2                                             
3    0 2015-07-21       0      4      3                                      
4    1 2014-02-19       3      0                                             

  item_9 item_10 item_11 item_12  basket_number  
0                                             0  
1                                             1  
2                                             2  
3                                             3  
4                                             4  


In [13]:
print("\nTest sample:")
print(test_df.head())



Test sample:
   uid       Date  item_1 item_2 item_3 item_4 item_5 item_6 item_7 item_8  \
0    0 2014-11-29       0                                                    
1    0 2014-12-15     113                                                    
2    0 2015-02-04     102                                                    
3    0 2015-07-21      64                                                    
4    1 2014-02-19      13                                                    

  item_9 item_10 item_11 item_12  basket_number  
0                                             0  
1                                             1  
2                                             2  
3                                             3  
4                                             4  


### 7. Generate 3 Files as Input for the Model:

In [14]:
print("Columns in final_df:", final_df.columns)
print("Columns in train_df:", train_df.columns)

if 'Date' not in train_df.columns or 'Date' not in test_df.columns:
    print("Error: 'Date' column is missing.")

Columns in final_df: Index(['uid', 'Date', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5',
       'item_6', 'item_7', 'item_8', 'item_9', 'item_10', 'item_11',
       'item_12'],
      dtype='object')
Columns in train_df: Index(['uid', 'Date', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5',
       'item_6', 'item_7', 'item_8', 'item_9', 'item_10', 'item_11', 'item_12',
       'basket_number'],
      dtype='object')


In [15]:
import os

# Check and create the directory if it does not exist
path_name = '/Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres'
output_dir = f"{path_name}"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [16]:
# # Training set - train_b2i
# # Drop the original 'uid' column 
# # Generate train_b2i file
# train_b2i = train_df[['basket_number', 'item_no']].explode('item_no').reset_index(drop=True)
# train_b2i['item_no'] = train_b2i['item_no'].astype(int)  # Ensure item_no is integers
# train_b2i['timestamp'] = train_df.loc[train_b2i['basket_number'], 'Date'].dt.strftime('%Y%m%d').values


# # Assign 'basket_number' as the first column
# if 'Date' in train_df.columns:
#     train_b2i['timestamp'] = train_df['Date'].dt.strftime('%Y%m%d')
# else:
#     raise KeyError("Date column is missing in train_df")

# columns = ['basket_number'] + [col for col in train_b2i.columns if col not in ['basket_number', 'timestamp']] + ['timestamp']
# train_b2i = train_b2i[columns]

# # Replace NaN with empty strings
# train_b2i = train_b2i.fillna('')

# # Convert all values to strings, handling empty values and non-string types
# def clean_and_convert(value):
#     if pd.isna(value) or value == '':
#         return ''
#     try:
#         if isinstance(value, (str, int)):
#             return str(int(value))
#         elif isinstance(value, float) and not pd.isna(value):
#             return str(int(value))
#         else:
#             return ''
#     except ValueError:
#         return ''

# for col in train_b2i.columns[1:]:
#     train_b2i[col] = train_b2i[col].apply(clean_and_convert)

# # Formatting data
# def format_row(row):
#     return ' '.join(str(x) for x in row if x != '')

# formatted_rows = train_b2i.apply(lambda row: ' '.join(str(x) for x in row if x != ''), axis=1)

# # Export the data as a text file without column names
# with open(f'{path_name}/train_b2i.txt', 'w') as file:
#     for row in formatted_rows:
#         file.write(row + '\n')

# print(f'Data exported to {path_name}/train_b2i.txt')

In [17]:
print("Columns in train_df:", train_df.columns)

Columns in train_df: Index(['uid', 'Date', 'item_1', 'item_2', 'item_3', 'item_4', 'item_5',
       'item_6', 'item_7', 'item_8', 'item_9', 'item_10', 'item_11', 'item_12',
       'basket_number'],
      dtype='object')


In [18]:
# train_b2i
train_b2i = train_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in train_b2i.columns if col != 'basket_number']
train_b2i = train_b2i[columns]

# Replace NaN with empty strings
train_b2i = train_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
def clean_and_convert(value):
    if pd.isna(value) or value == '':
        return ''
    try:
        if isinstance(value, (str, int)):
            return str(int(value))
        elif isinstance(value, float) and not pd.isna(value):
            return str(int(value))
        else:
            return ''
    except ValueError:
        return ''

for col in train_b2i.columns[1:]:
    train_b2i[col] = train_b2i[col].apply(clean_and_convert)

# Formatting data
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
train_b2i['timestamp'] = train_df['Date'].dt.strftime('%Y%m%d')

formatted_rows = train_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)

# formatted_rows = train_b2i.apply(format_row, axis=1)
def validate_timestamp(row):
    try:

        return row
    except ValueError:
        return None

formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  


# Export the data as a text file without column names
with open(f'{path_name}/train_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}\\train_b2i.txt')

Data exported to /Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres\train_b2i.txt


In [19]:
# # Extract and melt item columns for train_b2i
# item_columns = [col for col in train_df.columns if col.startswith('item_')]

# print(train_b2i.head())
# # Filter and clean
# train_b2i = train_b2i.dropna(subset=['item_no'])  # Drop NaN values
# train_b2i = train_b2i[train_b2i['item_no'] != '']  # Drop empty strings
# train_b2i['item_no'] = train_b2i['item_no'].astype(int)  # Convert to integers

# # Add timestamp
# train_b2i['timestamp'] = train_b2i['Date'].dt.strftime('%Y%m%d')

# # Export
# output_file = f'{path_name}/train_b2i.txt'
# train_b2i[['basket_number', 'item_no', 'timestamp']].to_csv(output_file, sep=' ', index=False, header=False)
# print(f"Data exported to {output_file}")

In [20]:
# # Test set - test_b2i
# # Drop the original 'uid' column 
# test_b2i = test_df.drop(columns=['uid'])

# test_b2i['timestamp'] = test_df['Date'].dt.strftime('%Y%m%d')

# # Assign 'basket_number' as the first column
# if 'basket_number' in test_b2i.columns:
#     columns = ['basket_number'] + [col for col in test_b2i.columns if col != 'basket_number']
#     test_b2i = test_b2i[columns]

# # Replace NaN with empty strings
# test_b2i = test_b2i.fillna('')

# # Convert all values to strings, handling empty values and non-string types
# for col in test_b2i.columns[1:]:  
#     test_b2i[col] = test_b2i[col].apply(clean_and_convert)

# # Formatting function to join values with space, ignoring empty strings
# def format_row(row):
#     return ' '.join(str(x) for x in row if x != '')

# # Formatting data
# formatted_rows = test_b2i.apply(lambda row: ' '.join(str(x) for x in row if x != ''), axis=1)

# # Export the data as a text file without column names
# with open(f'{path_name}/test_b2i.txt', 'w') as file:
#     for row in formatted_rows:
#         file.write(row + '\n')

# print(f'Data exported to {path_name}/test_b2i.txt')

In [21]:
# Test set - test_b2i
# Drop the original 'uid' column 
test_b2i = test_df.drop(columns=['uid'])

# Assign 'basket_number' as the first column
columns = ['basket_number'] + [col for col in test_b2i.columns if col != 'basket_number']
test_b2i = test_b2i[columns]

# Replace NaN with empty strings
test_b2i = test_b2i.fillna('')

# Convert all values to strings, handling empty values and non-string types
for col in test_b2i.columns[1:]:  
    test_b2i[col] = test_b2i[col].apply(clean_and_convert)

# Formatting function to join values with space, ignoring empty strings
def format_row(row):
    return ' '.join(str(x) for x in row if x != '')

# Add timestamps
test_b2i['timestamp'] = test_df['Date'].dt.strftime('%Y%m%d')

formatted_rows = test_b2i.apply(lambda row: ' '.join(str(x).strip() for x in row if x != ''), axis=1)
formatted_rows = formatted_rows.apply(validate_timestamp).dropna()  

# Formatting data
# formatted_rows = test_b2i.apply(format_row, axis=1)

# Export the data as a text file without column names
with open(f'{path_name}/test_b2i.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}\\test_b2i.txt')

Data exported to /Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres\test_b2i.txt


In [22]:
# # Extract and melt item columns for test_b2i
# item_columns = [col for col in test_df.columns if col.startswith('item_')]

# test_b2i = test_df.melt(
#     id_vars=['basket_number', 'Date'],
#     value_vars=item_columns,
#     var_name='item_column',
#     value_name='item_no'
# ).drop(columns=['item_column'])

# # Filter and clean
# test_b2i = test_b2i.dropna(subset=['item_no'])  # Drop NaN values
# test_b2i = test_b2i[test_b2i['item_no'] != '']  # Drop empty strings
# test_b2i['item_no'] = test_b2i['item_no'].astype(int)  # Convert to integers

# # Add timestamp
# test_b2i['timestamp'] = test_b2i['Date'].dt.strftime('%Y%m%d')

# # Export
# # output_file = f'{path_name}/test_b2i.txt'
# # test_b2i[['basket_number', 'item_no', 'timestamp']].to_csv(output_file, sep=' ', index=False, header=False)
# # print(f"Data exported to {output_file}")

In [23]:
# train_u2b
grouped_df = train_df.groupby('uid')['basket_number'].apply(list).reset_index()
expanded_df = grouped_df['basket_number'].apply(pd.Series)
expanded_df.columns = [f'basket_{i+1}' for i in expanded_df.columns]

# Concatenate the expanded columns with the original 'uid' column
train_u2b = pd.concat([grouped_df['uid'], expanded_df], axis=1)

# Replace NaN with empty strings
train_u2b = train_u2b.fillna('')

# Assign 'uid' as the first column
columns = ['uid'] + [col for col in train_u2b.columns if col != 'uid']
train_u2b = train_u2b[columns]

# Convert all values to strings, handling empty values and non-string types
for col in train_u2b.columns[1:]:
    train_u2b[col] = train_u2b[col].apply(clean_and_convert)

# Formatting data
formatted_rows = train_u2b.apply(format_row, axis=1)

# Export the data as a text file without column names
with open(f'{path_name}/train_u2b.txt', 'w') as file:
    for row in formatted_rows:
        file.write(row + '\n')

print(f'Data exported to {path_name}/train_u2b.txt')

Data exported to /Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres/train_u2b.txt


In [24]:
# # train_u2b
# # Ensure 'uid' and 'Date' columns exist in train_df
# if 'uid' not in train_df.columns or 'Date' not in train_df.columns:
#     raise KeyError("Error: 'uid' or 'Date' column is missing in train_df")

# # Group data by 'uid' and ensure 'basket_number' and 'Date' are sorted
# grouped_df = train_df.groupby('uid', group_keys=False).apply(
#     lambda x: x.sort_values(by='Date')
# ).reset_index(drop=True)

# # Check if 'basket_number' column exists
# if 'basket_number' not in grouped_df.columns:
#     raise KeyError("Error: 'basket_number' column is missing in grouped_df after grouping")

# # Expand the 'basket_number' column into separate columns
# expanded_basket_df = grouped_df['basket_number'].apply(pd.Series)
# expanded_basket_df.columns = [f'basket_{i+1}' for i in expanded_basket_df.columns]

# # Combine 'uid' and expanded basket columns (exclude 'Date' to remove timestamps)
# train_u2b = pd.concat([grouped_df[['uid']], expanded_basket_df], axis=1)

# # Replace NaN with empty strings
# train_u2b = train_u2b.fillna('')

# # Convert all values to strings
# for col in train_u2b.columns:
#     train_u2b[col] = train_u2b[col].astype(str)

# # Formatting each row into the desired format
# def format_row(row):
#     """
#     Format each row into a single space-separated string.
#     Each row includes 'uid' and all basket entries, without timestamps.
#     """
#     uid = row['uid']
#     baskets = [row[col] for col in train_u2b.columns if col.startswith('basket_') and row[col].strip()]
#     return f"{uid} " + " ".join(baskets)

# formatted_rows = train_u2b.apply(format_row, axis=1)

# # Export the data to a text file
# output_dir = f"{path_name}"
# os.makedirs(output_dir, exist_ok=True)  # Create the directory if it doesn't exist

# output_file = os.path.join(output_dir, "train_u2b.txt")
# with open(output_file, 'w', encoding='utf-8') as file:
#     file.write('\n'.join(formatted_rows) + '\n')

# print(f"Data exported to {output_file}")

In [25]:
# # test_u2b.txt
# # Ensure 'uid' and 'basket_number' columns exist in test_df
# if 'uid' not in test_df.columns or 'basket_number' not in test_df.columns:
#     raise KeyError("Error: 'uid' or 'basket_number' column is missing in test_df")

# # Extract necessary columns and sort by uid and basket_number
# sorted_test_df = test_df[['uid', 'basket_number']].sort_values(by=['uid', 'basket_number'])

# # Convert all values to integers
# sorted_test_df['uid'] = sorted_test_df['uid'].astype(int)
# sorted_test_df['basket_number'] = sorted_test_df['basket_number'].astype(int)

# # Format each row into "uid basket_number"
# formatted_rows = sorted_test_df.apply(lambda row: f"{row['uid']} {row['basket_number']}", axis=1)

# # Export the formatted rows to test_u2b.txt
# output_file = os.path.join(output_dir, "test_u2b.txt")
# with open(output_file, 'w', encoding='utf-8') as file:
#     file.write('\n'.join(formatted_rows) + '\n')

# print(f"Data exported to {output_file}")


In [26]:
# test_u2b
grouped_df_test = test_df.groupby('uid')['basket_number'].apply(list).reset_index()
expanded_df_test = grouped_df_test['basket_number'].apply(pd.Series)
expanded_df_test.columns = [f'basket_{i+1}' for i in expanded_df_test.columns]

test_u2b = pd.concat([grouped_df_test['uid'], expanded_df_test], axis=1)
test_u2b = test_u2b.fillna('')

columns = ['uid'] + [col for col in test_u2b.columns if col != 'uid']
test_u2b = test_u2b[columns]

for col in test_u2b.columns[1:]:
    test_u2b[col] = test_u2b[col].apply(clean_and_convert)

formatted_rows_test = test_u2b.apply(format_row, axis=1)

output_file = os.path.join(output_dir, "test_u2b.txt")
with open(f'{path_name}/test_u2b.txt', 'w') as file:
    for row in formatted_rows_test:
        file.write(row + '\n')
print(f"Data exported to {output_file}")

Data exported to /Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres/test_u2b.txt


In [27]:
print("Output directory:", output_dir)

Output directory: /Users/jynn/Desktop/spring2025/4737 ai application in finance/2024 materials/FinalSubmission/Code-TGN/DataPreparationandTransformation/groceries_data/data/grocerieswithres


In [28]:
# # Read the basket_number list from test_u2b.txt
# test_u2b_file = os.path.join(output_dir, "test_u2b.txt")
# test_baskets_in_u2b = set()

# with open(test_u2b_file, 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         # The format of test_u2b is: "uid basket_number"
#         # uid is parts[0], basket_number is parts[1]
#         if len(parts) >= 2:
#             basket_num = parts[1]
#             if basket_num.isdigit():
#                 test_baskets_in_u2b.add(int(basket_num))

# # Read test_b2i.txt and filter out baskets not present in test_u2b
# test_b2i_file = os.path.join(output_dir, "test_b2i.txt")
# cleaned_lines = []
# with open(test_b2i_file, 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         # The format of test_b2i is usually: "basket_number item_no timestamp"
#         if len(parts) == 3:
#             basket_num_str = parts[0]
#             if basket_num_str.isdigit():
#                 basket_num = int(basket_num_str)
#                 # Keep only baskets that are present in test_u2b
#                 if basket_num in test_baskets_in_u2b:
#                     cleaned_lines.append(line.strip())

# # Write the cleaned data back to test_b2i.txt
# with open(test_b2i_file, 'w', encoding='utf-8') as f:
#     for line in cleaned_lines:
#         f.write(line + '\n')

# print("Cleaning complete: All baskets in test_b2i.txt now correspond to users in test_u2b.")

In [29]:
# # Read the basket_number list from test_u2b.txt
# test_u2b_file = os.path.join(output_dir, "test_u2b.txt")
# test_baskets_in_u2b = set()

# with open(test_u2b_file, 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         # The format of test_u2b is: "uid basket_number"
#         # uid is parts[0], basket_number is parts[1]
#         if len(parts) >= 2:
#             basket_num = parts[1:-1]
#             for num in basket_num:
#                 if num.isdigit():
#                     test_baskets_in_u2b.add(int(num))

# # Read test_b2i.txt and filter out baskets not present in test_u2b
# test_b2i_file = os.path.join(output_dir, "test_b2i.txt")
# cleaned_lines = []
# with open(test_b2i_file, 'r', encoding='utf-8') as f:
#     for line in f:
#         parts = line.strip().split()
#         # The format of test_b2i is usually: "basket_number item_no timestamp"
#         if len(parts) >= 3:
#             basket_num_str = parts[0]
#             if basket_num_str.isdigit():
#                 basket_num = int(basket_num_str)
#                 # Keep only baskets that are present in test_u2b
#                 if basket_num in test_baskets_in_u2b:
#                     cleaned_lines.append(line.strip())

# # Write the cleaned data back to test_b2i.txt
# with open(test_b2i_file, 'w', encoding='utf-8') as f:
#     for line in cleaned_lines:
#         f.write(line + '\n')

# print("Cleaning complete: All baskets in test_b2i.txt now correspond to users in test_u2b.")